# Predictive Age-of-Information-Aware Hybrid Whale–Ant Optimization (AoI-WAOR)

End-to-end, reproducible simulation implementation for dynamic 6G routing. The notebook includes:
- Dynamic wireless topology and time-varying link conditions
- Predictive AoI-aware relay selection
- Whale Optimization Algorithm (WOA) route search
- Ant Colony Optimization (ACO) route construction
- Cooperative whale–ant information exchange and age-weighted pheromone updates
- Event-triggered route maintenance
- Baseline routing comparisons, metrics, and plots

**Simulation note:** Results generated by this notebook are simulation outputs, not measurements from a deployed 6G network. Adjust the configuration to match the assumptions and scale of a target experiment.

## 1. Imports and reproducibility

In [ ]:
import os
import math
import random
import heapq
from dataclasses import dataclass
from collections import defaultdict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

SAVE_DIR = "rslt"
os.makedirs(SAVE_DIR, exist_ok=True)

plt.rcParams.update({
    "figure.figsize": (9, 5),
    "figure.dpi": 120,
    "axes.grid": True,
    "grid.alpha": 0.25,
    "font.size": 11
})

## 2. Network and algorithm configuration

In [ ]:
@dataclass
class Config:
    n_nodes: int = 30
    area_size: float = 1000.0
    radio_range: float = 430.0
    episodes: int = 60
    packets_per_episode: int = 18
    source: int = 0
    destination: int = 29
    population: int = 18
    iterations: int = 22
    ants: int = 18
    alpha: float = 1.0
    beta: float = 2.0
    evaporation: float = 0.22
    q_pheromone: float = 1.0
    whale_probability: float = 0.45
    aoi_weight: float = 0.38
    delay_weight: float = 0.27
    reliability_weight: float = 0.25
    congestion_weight: float = 0.10
    maintenance_threshold: float = 0.22
    max_hops: int = 12
    packet_bits: int = 12000
    bandwidth_hz: float = 2e6
    tx_power_w: float = 0.1
    noise_w: float = 1e-10
    slot_ms: float = 10.0
    max_link_delay_ms: float = 18.0

CFG = Config()

## 3. Dynamic 6G network environment

In [ ]:
class Dynamic6GNetwork:
    def __init__(self, cfg):
        self.cfg = cfg
        self.positions = np.random.uniform(0, cfg.area_size, (cfg.n_nodes, 2))
        self.queue_load = np.random.uniform(0.05, 0.35, cfg.n_nodes)
        self.link_state = {}
        self.time = 0
        self._build_links()

    def _build_links(self):
        self.neighbors = {i: [] for i in range(self.cfg.n_nodes)}
        for i in range(self.cfg.n_nodes):
            for j in range(i + 1, self.cfg.n_nodes):
                d = np.linalg.norm(self.positions[i] - self.positions[j])
                if d <= self.cfg.radio_range:
                    self.neighbors[i].append(j)
                    self.neighbors[j].append(i)
                    self.link_state[(i, j)] = self._new_link()
                    self.link_state[(j, i)] = self.link_state[(i, j)]

    def _new_link(self):
        return {
            "capacity_mbps": float(np.random.uniform(40, 180)),
            "success": float(np.random.uniform(0.91, 0.999)),
            "delay_ms": float(np.random.uniform(1.0, 5.0)),
            "aoi_ms": float(np.random.uniform(4.0, 25.0))
        }

    def step(self):
        self.time += 1
        self.queue_load = np.clip(
            0.82 * self.queue_load + 0.18 * np.random.uniform(0.02, 0.65, self.cfg.n_nodes),
            0.01, 0.95
        )
        for key in list(self.link_state):
            if key[0] < key[1]:
                old = self.link_state[key]
                updated = {
                    "capacity_mbps": float(np.clip(old["capacity_mbps"] + np.random.normal(0, 8), 20, 200)),
                    "success": float(np.clip(old["success"] + np.random.normal(0, 0.012), 0.82, 0.999)),
                    "delay_ms": float(np.clip(old["delay_ms"] + np.random.normal(0, 0.45), 0.8, 12)),
                    "aoi_ms": float(np.clip(old["aoi_ms"] + np.random.normal(0, 1.8), 2, 45))
                }
                self.link_state[key] = updated
                self.link_state[(key[1], key[0])] = updated

        # Mobility: small position changes may alter connectivity.
        self.positions += np.random.normal(0, 5.0, self.positions.shape)
        self.positions = np.clip(self.positions, 0, self.cfg.area_size)
        self._refresh_topology()

    def _refresh_topology(self):
        self.neighbors = {i: [] for i in range(self.cfg.n_nodes)}
        for i in range(self.cfg.n_nodes):
            for j in range(i + 1, self.cfg.n_nodes):
                d = np.linalg.norm(self.positions[i] - self.positions[j])
                if d <= self.cfg.radio_range:
                    self.neighbors[i].append(j)
                    self.neighbors[j].append(i)
                    if (i, j) not in self.link_state:
                        self.link_state[(i, j)] = self._new_link()
                        self.link_state[(j, i)] = self.link_state[(i, j)]

    def link(self, u, v):
        return self.link_state.get((u, v))

    def path_metrics(self, path):
        if path is None or len(path) < 2:
            return None
        delay = 0.0
        success = 1.0
        aoi = 0.0
        congestion = 0.0
        for u, v in zip(path[:-1], path[1:]):
            link = self.link(u, v)
            if link is None:
                return None
            delay += link["delay_ms"] + self.cfg.packet_bits / (link["capacity_mbps"] * 1e6) * 1000
            success *= link["success"]
            aoi += link["aoi_ms"] + link["delay_ms"]
            congestion += 0.5 * (self.queue_load[u] + self.queue_load[v])
        hops = len(path) - 1
        return {
            "delay_ms": delay,
            "pdr": 100 * success,
            "aoi_ms": aoi,
            "congestion": congestion / hops,
            "hops": hops
        }

    def feasible_neighbors(self, node, visited):
        return [v for v in self.neighbors[node] if v not in visited]

## 4. Predictive AoI-aware relay scoring and route objective

In [ ]:
def predict_link_aoi(net, u, v):
    link = net.link(u, v)
    if link is None:
        return np.inf
    # One-step prediction: current freshness plus expected transmission and queueing impact.
    queue_penalty = 4.0 * 0.5 * (net.queue_load[u] + net.queue_load[v])
    return link["aoi_ms"] + link["delay_ms"] + queue_penalty

def relay_utility(net, u, v, cfg):
    link = net.link(u, v)
    if link is None:
        return 0.0
    pred_aoi = predict_link_aoi(net, u, v)
    score = (
        0.42 * (1.0 / (1.0 + pred_aoi / 20.0)) +
        0.33 * link["success"] +
        0.15 * (1.0 / (1.0 + link["delay_ms"] / 5.0)) +
        0.10 * (1.0 - 0.5 * (net.queue_load[u] + net.queue_load[v]))
    )
    return max(score, 1e-8)

def route_cost(net, path, cfg):
    m = net.path_metrics(path)
    if m is None:
        return 1e6
    aoi = m["aoi_ms"]
    delay = m["delay_ms"]
    reliability_loss = 100.0 - m["pdr"]
    congestion = m["congestion"] * 100.0
    # Normalize terms to comparable scales.
    cost = (
        cfg.aoi_weight * aoi / 100.0 +
        cfg.delay_weight * delay / 100.0 +
        cfg.reliability_weight * reliability_loss / 100.0 +
        cfg.congestion_weight * congestion / 100.0
    )
    if m["hops"] > cfg.max_hops:
        cost += 2.0 * (m["hops"] - cfg.max_hops)
    return float(cost)

def route_is_valid(net, path, cfg):
    if path is None or len(path) < 2 or len(path) - 1 > cfg.max_hops:
        return False
    if path[0] != cfg.source or path[-1] != cfg.destination:
        return False
    return all(net.link(u, v) is not None for u, v in zip(path[:-1], path[1:]))

## 5. Route construction utilities

In [ ]:
def shortest_path_by_weight(net, cfg, weight="delay"):
    start, goal = cfg.source, cfg.destination
    pq = [(0.0, start, [start])]
    best = {start: 0.0}
    while pq:
        cost, node, path = heapq.heappop(pq)
        if node == goal:
            return path
        if cost > best.get(node, np.inf):
            continue
        for nxt in net.neighbors[node]:
            if nxt in path:
                continue
            link = net.link(node, nxt)
            if weight == "delay":
                edge = link["delay_ms"]
            elif weight == "aoi":
                edge = predict_link_aoi(net, node, nxt)
            elif weight == "reliability":
                edge = -math.log(max(link["success"], 1e-8))
            else:
                edge = link["delay_ms"] + 5 * (1 - link["success"])
            nc = cost + edge
            if nc < best.get(nxt, np.inf):
                best[nxt] = nc
                heapq.heappush(pq, (nc, nxt, path + [nxt]))
    return None

def repair_path(net, path, cfg):
    if path is None:
        return None
    clean = [path[0]]
    for node in path[1:]:
        if node not in clean and net.link(clean[-1], node) is not None:
            clean.append(node)
        else:
            break
    if clean[-1] == cfg.destination:
        return clean
    # Greedy completion using predictive relay utility.
    current = clean[-1]
    while current != cfg.destination and len(clean) <= cfg.max_hops:
        candidates = net.feasible_neighbors(current, set(clean))
        if not candidates:
            return None
        candidates.sort(key=lambda v: (
            np.linalg.norm(net.positions[v] - net.positions[cfg.destination]) -
            80 * relay_utility(net, current, v, cfg)
        ))
        nxt = candidates[0]
        clean.append(nxt)
        current = nxt
    return clean if clean[-1] == cfg.destination else None

def random_feasible_path(net, cfg):
    for _ in range(80):
        path = [cfg.source]
        while path[-1] != cfg.destination and len(path) <= cfg.max_hops:
            candidates = net.feasible_neighbors(path[-1], set(path))
            if not candidates:
                break
            # Destination proximity with predictive AoI-aware relay bias.
            weights = []
            for v in candidates:
                dist = np.linalg.norm(net.positions[v] - net.positions[cfg.destination])
                weights.append(relay_utility(net, path[-1], v, cfg) / (1 + dist / cfg.area_size))
            weights = np.asarray(weights, dtype=float)
            weights /= weights.sum()
            path.append(int(np.random.choice(candidates, p=weights)))
        if path[-1] == cfg.destination and route_is_valid(net, path, cfg):
            return path
    return shortest_path_by_weight(net, cfg, "delay")

## 6. Whale Optimization Algorithm (WOA) route search

In [ ]:
def path_to_vector(path, cfg):
    internal = path[1:-1] if path else []
    vals = np.zeros(cfg.max_hops - 1)
    vals[:min(len(internal), len(vals))] = internal[:len(vals)]
    return vals

def vector_to_path(vec, net, cfg):
    # Convert a continuous whale position to a simple feasible node sequence.
    candidates = np.clip(np.rint(vec).astype(int), 0, cfg.n_nodes - 1).tolist()
    path = [cfg.source]
    for node in candidates:
        if node == cfg.destination:
            break
        if node not in path and net.link(path[-1], node) is not None:
            path.append(node)
        if len(path) >= cfg.max_hops:
            break
    if net.link(path[-1], cfg.destination) is not None and path[-1] != cfg.destination:
        path.append(cfg.destination)
    else:
        completed = repair_path(net, path, cfg)
        return completed
    return path if route_is_valid(net, path, cfg) else repair_path(net, path, cfg)

def woa_optimize(net, cfg, seed_paths=None):
    pop = []
    if seed_paths:
        pop.extend([p for p in seed_paths if route_is_valid(net, p, cfg)])
    while len(pop) < cfg.population:
        p = random_feasible_path(net, cfg)
        if p is not None:
            pop.append(p)
        else:
            break
    if not pop:
        return shortest_path_by_weight(net, cfg, "delay")

    X = np.array([path_to_vector(p, cfg) for p in pop])
    while len(X) < cfg.population:
        X = np.vstack([X, path_to_vector(random_feasible_path(net, cfg), cfg)])

    def decode(x):
        p = vector_to_path(x, net, cfg)
        return p if p is not None and route_is_valid(net, p, cfg) else shortest_path_by_weight(net, cfg, "delay")

    fitness = np.array([route_cost(net, decode(x), cfg) for x in X])
    best_idx = int(np.argmin(fitness))
    best_vec = X[best_idx].copy()
    best_path = decode(best_vec)

    for t in range(cfg.iterations):
        a = 2 - 2 * t / max(cfg.iterations - 1, 1)
        for i in range(len(X)):
            r1, r2 = np.random.rand(), np.random.rand()
            A = 2 * a * r1 - a
            C = 2 * r2
            p = np.random.rand()
            if p < 0.5:
                if abs(A) < 1:
                    D = abs(C * best_vec - X[i])
                    new = best_vec - A * D
                else:
                    rand_vec = X[np.random.randint(len(X))]
                    D = abs(C * rand_vec - X[i])
                    new = rand_vec - A * D
            else:
                l = np.random.uniform(-1, 1)
                D = abs(best_vec - X[i])
                new = D * np.exp(l) * np.cos(2 * np.pi * l) + best_vec
            new = np.clip(new, 0, cfg.n_nodes - 1)
            candidate = decode(new)
            c = route_cost(net, candidate, cfg)
            if c < fitness[i]:
                X[i] = path_to_vector(candidate, cfg)
                fitness[i] = c
            if c < route_cost(net, best_path, cfg):
                best_path = candidate
                best_vec = path_to_vector(candidate, cfg)
    return best_path

## 7. Ant Colony Optimization with age-weighted pheromone reinforcement

In [ ]:
class AgeAwareACO:
    def __init__(self, cfg):
        self.cfg = cfg
        self.pheromone = defaultdict(lambda: 1.0)

    def construct_ant(self, net, whale_hint=None):
        path = [self.cfg.source]
        while path[-1] != self.cfg.destination and len(path) <= self.cfg.max_hops:
            current = path[-1]
            candidates = net.feasible_neighbors(current, set(path))
            if not candidates:
                return None
            desirability = []
            for nxt in candidates:
                link = net.link(current, nxt)
                tau = self.pheromone[(current, nxt)]
                eta = relay_utility(net, current, nxt, self.cfg)
                if whale_hint and len(whale_hint) > 1 and nxt in whale_hint:
                    eta *= 1.35
                desirability.append((tau ** self.cfg.alpha) * (eta ** self.cfg.beta))
            probs = np.asarray(desirability, dtype=float)
            probs /= probs.sum()
            path.append(int(np.random.choice(candidates, p=probs)))
        return path if route_is_valid(net, path, self.cfg) else None

    def update(self, net, paths):
        for edge in list(self.pheromone):
            self.pheromone[edge] *= (1 - self.cfg.evaporation)
        valid = [p for p in paths if route_is_valid(net, p, self.cfg)]
        for path in valid:
            metrics = net.path_metrics(path)
            # Freshness, reliability, and delay jointly govern pheromone deposit.
            freshness = 1 / (1 + metrics["aoi_ms"] / 50)
            reliability = metrics["pdr"] / 100
            latency = 1 / (1 + metrics["delay_ms"] / 50)
            deposit = self.cfg.q_pheromone * freshness * reliability * latency
            deposit /= max(route_cost(net, path, self.cfg), 1e-4)
            for u, v in zip(path[:-1], path[1:]):
                self.pheromone[(u, v)] += deposit

    def optimize(self, net, whale_hint=None):
        all_paths = []
        for _ in range(self.cfg.ants):
            p = self.construct_ant(net, whale_hint)
            if p is not None:
                all_paths.append(p)
        self.update(net, all_paths)
        if not all_paths:
            return None
        return min(all_paths, key=lambda p: route_cost(net, p, self.cfg))

## 8. Cooperative Whale–Ant Optimization (AoI-WAOR)

In [ ]:
class AoI_WAOR:
    def __init__(self, cfg):
        self.cfg = cfg
        self.aco = AgeAwareACO(cfg)
        self.current_path = None
        self.last_cost = np.inf
        self.maintenance_events = 0

    def optimize(self, net):
        ant_seed = self.aco.optimize(net, self.current_path)
        seeds = [p for p in [self.current_path, ant_seed] if p is not None]
        whale_path = woa_optimize(net, self.cfg, seeds)
        # Reciprocal information exchange: ant search receives whale route as a hint.
        ant_refined = self.aco.optimize(net, whale_path)
        candidates = [p for p in [whale_path, ant_seed, ant_refined] if p is not None and route_is_valid(net, p, self.cfg)]
        if candidates:
            self.current_path = min(candidates, key=lambda p: route_cost(net, p, self.cfg))
            self.last_cost = route_cost(net, self.current_path, self.cfg)
        return self.current_path

    def event_triggered_maintenance(self, net):
        if self.current_path is None:
            self.optimize(net)
            return True
        current_metrics = net.path_metrics(self.current_path)
        current_cost = route_cost(net, self.current_path, self.cfg)
        invalid = not route_is_valid(net, self.current_path, self.cfg)
        degradation = max(0.0, (current_cost - self.last_cost) / max(abs(self.last_cost), 1e-6))
        # Trigger when path breaks or normalized objective degradation exceeds threshold.
        if invalid or degradation > self.cfg.maintenance_threshold:
            self.maintenance_events += 1
            self.optimize(net)
            return True
        return False

## 9. Routing baselines

In [ ]:
def select_route(net, method, cfg, hybrid=None):
    if method == "AoI-WAOR":
        return hybrid.optimize(net)
    if method == "Shortest Delay":
        return shortest_path_by_weight(net, cfg, "delay")
    if method == "AoI-Greedy":
        return shortest_path_by_weight(net, cfg, "aoi")
    if method == "Reliability":
        return shortest_path_by_weight(net, cfg, "reliability")
    if method == "ACO":
        return AgeAwareACO(cfg).optimize(net)
    if method == "WOA":
        return woa_optimize(net, cfg)
    return shortest_path_by_weight(net, cfg, "delay")

## 10. Simulation runner

In [ ]:
def run_experiment(cfg, methods=None):
    if methods is None:
        methods = ["Shortest Delay", "AoI-Greedy", "Reliability", "ACO", "WOA", "AoI-WAOR"]

    results = []
    network = Dynamic6GNetwork(cfg)
    hybrids = {m: AoI_WAOR(cfg) for m in methods if m == "AoI-WAOR"}
    routes = {m: None for m in methods}

    for ep in range(cfg.episodes):
        if ep > 0:
            network.step()

        for method in methods:
            if method == "AoI-WAOR":
                hybrid = hybrids[method]
                if hybrid.current_path is None:
                    path = hybrid.optimize(network)
                else:
                    triggered = hybrid.event_triggered_maintenance(network)
                    path = hybrid.current_path
            else:
                path = select_route(network, method, cfg)
            routes[method] = path

            if path is None or not route_is_valid(network, path, cfg):
                metric = {
                    "aoi_ms": np.nan, "delay_ms": np.nan,
                    "pdr": 0.0, "congestion": np.nan, "hops": np.nan
                }
            else:
                metric = network.path_metrics(path)

            # Packet-level delivery sampling provides a stochastic PDR estimate.
            if path is not None and route_is_valid(network, path, cfg):
                p_success = metric["pdr"] / 100
                delivered = np.random.binomial(cfg.packets_per_episode, p_success)
                observed_pdr = 100 * delivered / cfg.packets_per_episode
            else:
                observed_pdr = 0.0

            results.append({
                "Episode": ep + 1,
                "Method": method,
                "AoI (ms)": metric["aoi_ms"],
                "End-to-End Delay (ms)": metric["delay_ms"],
                "PDR (%)": observed_pdr,
                "Congestion": metric["congestion"],
                "Hop Count": metric["hops"],
                "Route Cost": route_cost(network, path, cfg) if path is not None and route_is_valid(network, path, cfg) else np.nan,
                "Maintenance Events": hybrids[method].maintenance_events if method == "AoI-WAOR" else 0
            })
    return pd.DataFrame(results), routes, network

results, final_routes, final_network = run_experiment(CFG)
results.head()

## 11. Performance summary

In [ ]:
summary = results.groupby("Method").agg({
    "AoI (ms)": "mean",
    "End-to-End Delay (ms)": "mean",
    "PDR (%)": "mean",
    "Congestion": "mean",
    "Hop Count": "mean",
    "Route Cost": "mean",
    "Maintenance Events": "max"
}).reset_index()

summary = summary.sort_values("Route Cost")
display(summary.round(4))

## 12. Convergence and comparative performance plots

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))
metrics_to_plot = [
    ("AoI (ms)", "Average Age of Information (ms)"),
    ("End-to-End Delay (ms)", "End-to-End Delay (ms)"),
    ("PDR (%)", "Packet Delivery Ratio (%)"),
    ("Route Cost", "Weighted Routing Objective")
]
for ax, (col, title) in zip(axes.ravel(), metrics_to_plot):
    for method, group in results.groupby("Method"):
        ax.plot(group["Episode"], group[col].rolling(5, min_periods=1).mean(), label=method, linewidth=2)
    ax.set_title(title, fontweight="bold")
    ax.set_xlabel("Episode")
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.8))
plot_cols = ["AoI (ms)", "End-to-End Delay (ms)", "PDR (%)"]
for ax, col in zip(axes, plot_cols):
    vals = [results.loc[results["Method"] == m, col].dropna().values for m in summary["Method"]]
    ax.boxplot(vals, labels=summary["Method"], showfliers=False)
    ax.set_title(col, fontweight="bold")
    ax.tick_params(axis="x", rotation=35)
plt.tight_layout()
plt.show()

## 13. Final selected route visualization

In [ ]:
hybrid_path = final_routes.get("AoI-WAOR")
if hybrid_path:
    pos = final_network.positions
    plt.figure(figsize=(8, 7))
    plt.scatter(pos[:, 0], pos[:, 1], s=45, alpha=0.75)
    for i, (x, y) in enumerate(pos):
        plt.text(x + 8, y + 8, str(i), fontsize=8)
    for u in range(CFG.n_nodes):
        for v in final_network.neighbors[u]:
            if u < v:
                plt.plot([pos[u, 0], pos[v, 0]], [pos[u, 1], pos[v, 1]], alpha=0.12, linewidth=0.7)
    for u, v in zip(hybrid_path[:-1], hybrid_path[1:]):
        plt.plot([pos[u, 0], pos[v, 0]], [pos[u, 1], pos[v, 1]], linewidth=3, marker="o")
    plt.scatter(pos[CFG.source, 0], pos[CFG.source, 1], s=180, marker="s", label="Source")
    plt.scatter(pos[CFG.destination, 0], pos[CFG.destination, 1], s=180, marker="*", label="Destination")
    plt.title("AoI-WAOR Selected End-to-End Route", fontweight="bold")
    plt.xlabel("X position (m)")
    plt.ylabel("Y position (m)")
    plt.legend()
    plt.tight_layout()
    plt.show()
    print("Selected route:", hybrid_path)
else:
    print("No feasible AoI-WAOR route was found in the final episode.")

## 14. Statistical comparison (episode-wise paired tests)

In [ ]:
def permutation_test_paired(x, y, n_perm=10000, seed=42):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    mask = np.isfinite(x) & np.isfinite(y)
    d = x[mask] - y[mask]
    if len(d) < 2:
        return np.nan
    rng = np.random.default_rng(seed)
    observed = abs(np.mean(d))
    signs = rng.choice([-1, 1], size=(n_perm, len(d)))
    null = np.abs((signs * d).mean(axis=1))
    return (np.sum(null >= observed) + 1) / (n_perm + 1)

stat_rows = []
for baseline in [m for m in results["Method"].unique() if m != "AoI-WAOR"]:
    for metric in ["AoI (ms)", "End-to-End Delay (ms)", "PDR (%)"]:
        a = results[results["Method"] == "AoI-WAOR"].sort_values("Episode")[metric].values
        b = results[results["Method"] == baseline].sort_values("Episode")[metric].values
        stat_rows.append({
            "Comparison": f"AoI-WAOR vs {baseline}",
            "Metric": metric,
            "Paired permutation p-value": permutation_test_paired(a, b)
        })
statistical_results = pd.DataFrame(stat_rows)
display(statistical_results.round(6))

## 15. Export simulation results

In [ ]:
summary.to_csv(os.path.join(SAVE_DIR, "AoI_WAOR_performance_summary.csv"), index=False)
results.to_csv(os.path.join(SAVE_DIR, "AoI_WAOR_episode_results.csv"), index=False)
statistical_results.to_csv(os.path.join(SAVE_DIR, "AoI_WAOR_statistical_tests.csv"), index=False)

with pd.ExcelWriter(os.path.join(SAVE_DIR, "AoI_WAOR_results.xlsx")) as writer:
    summary.to_excel(writer, sheet_name="Performance Summary", index=False)
    results.to_excel(writer, sheet_name="Episode Results", index=False)
    statistical_results.to_excel(writer, sheet_name="Statistical Tests", index=False)

print("Results exported to:", os.path.abspath(SAVE_DIR))

## 16. Suggested experiment extensions

For a larger evaluation, vary node density, mobility, offered traffic, radio range, and channel volatility independently. Repeat each configuration over multiple random seeds and report mean ± standard deviation and confidence intervals. For URLLC evaluation, explicitly define the target latency and reliability thresholds in the experiment configuration before claiming that a run satisfies them.